📒 [가이드] 모델 로딩 및 추론 표준 코드

작성자: 박제혁

목적: Hydra 설정과 ModelFactory를 사용하여 모델을 로드하고, 추론을 수행하는 방법을 공유합니다. 각자 담당한 노드(Solver, Critic 등) 구현 시 참고하세요.

In [1]:
# [Cell 1] 환경 설정 및 임포트
import sys
import os

# 1. 프로젝트 루트 경로 추가 (notebooks 폴더 기준 상위 폴더)
# 이걸 해줘야 src 패키지를 불러올 수 있습니다.
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. 필요한 모듈 임포트
from src.utils.config_loader import load_config
from src.agent.nodes.base import BaseLLMNode

# 3. 설정 로드 (우리가 만든 유틸 함수 사용)
cfg = load_config()

print("✅ 설정 및 라이브러리 로드 완료!")
print(f"🔹 사용 모델: {cfg.model.main_solver.path}")

✅ 설정 및 라이브러리 로드 완료!
🔹 사용 모델: unsloth/Qwen3-32B-bnb-4bit


In [2]:
# [Cell 2] 나만의 노드 만들기 (팀원들이 해야 할 작업)
# 복잡한 모델 로딩 코드는 사라지고, 딱 '프롬프트'와 '로직'만 남습니다.

class MyTestNode(BaseLLMNode):
    def __init__(self, config):
        # 1. 부모 클래스 초기화 (모델 로딩은 여기서 다 알아서 함)
        # model_name="main_solver" 또는 "sub_solver" 선택 가능
        super().__init__(config, model_name="main_solver")
        
        # 2. 테스트용 프롬프트 템플릿 정의
        # (실제로는 config.prompt.xxx.template 에서 가져오면 됩니다)
        self.template = """
        당신은 친절한 AI 조교입니다.
        사용자의 질문에 대해 핵심만 요약해서 답변하세요.
        
        질문: {question}
        요약 답변:
        """

    def __call__(self, state: dict) -> dict:
        """LangGraph가 실행할 함수"""
        print(f"▶️ 노드 실행 중... 질문: {state['question']}")
        
        # 3. generate 함수 호출 (핵심!)
        # 템플릿에 있는 {question} 변수만 채워주면 끝
        answer = self.generate(self.template, question=state["question"])
        
        return {"answer": answer}

print("✅ MyTestNode 클래스 정의 완료! (모델은 아직 로드 안 됨)")

✅ MyTestNode 클래스 정의 완료! (모델은 아직 로드 안 됨)


In [ ]:
# [Cell 3] 노드 실행 테스트
# 실제 모델이 메모리에 올라가고 추론이 진행됩니다.

# 1. 노드 생성 (이 시점에 모델이 로딩됨 - 시간 소요)
print("⏳ 모델 로딩 시작... (잠시만 기다려주세요)")
node = MyTestNode(cfg)
print("✅ 모델 로딩 완료!")

# 2. 가짜 상태(State) 데이터 생성
dummy_state = {
    "question": "2024년 수능 영어 영역의 난이도가 어땠는지 알려줘."
}

# 3. 노드 실행 (추론)
result = node(dummy_state)

# 4. 결과 확인
print("\n" + "="*30)
print(f"🤖 결과:\n{result['answer']}")
print("="*30)

⏳ 모델 로딩 시작... (잠시만 기다려주세요)
🔧 [MyTestNode] 초기화 중... (사용 모델: main_solver)
🔄 [Loader] 모델 로딩 시작: Qwen3-32B-4bit (unsloth/Qwen3-32B-bnb-4bit)
   ↳ ⚡ 양자화 설정 적용 중...


/data/ephemeral/home/REPO_Jehyeok/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ [Loader] 로딩 완료!
✅ 모델 로딩 완료!
▶️ 노드 실행 중... 질문: 2024년 수능 영어 영역의 난이도가 어땠는지 알려줘.
